In [4]:
import pandas as pd
import numpy as np

In [5]:
# Load the .xyz file
df = pd.read_csv("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/sedthk-m.xyz", 
                 sep='\s+',   # whitespace delimited
                 header=None,
                 names=['lon', 'lat', 'thickness'])

In [ ]:
# Check zero/null distribution geographically
zeros = df[df['thickness'] <= 0]
print(f"Zero/negative thickness locations: {len(zeros)}")
print(f"  Lat range of zeros: {zeros['lat'].min():.1f} – {zeros['lat'].max():.1f}")
print(f"  Lon range of zeros: {zeros['lon'].min():.1f} – {zeros['lon'].max():.1f}")

# Convert to km for interpretability
df['thickness_km'] = df['thickness'] / 1000

# Check each patch
patches = {
    "Kanto_Japan":      (138.5, 34.5, 141.5, 37.2),
    "Tohoku_Japan":     (140.5, 37.5, 143.5, 40.5),
    "Central_Chile":    (-72.5, -36.5, -69.5, -33.5),
    "Central_Turkey":   (35.5, 36.5, 39.0, 39.0),
    "Nepal":            (83.5, 27.0, 86.5, 29.7),
    "North_Island_NZ":  (174.5, -40.5, 178.0, -37.5),
    "Sumatra":          (100.5, -5.5, 104.5, -2.0),
    "Kutch_India":      (68.5, 21.5, 72.0, 24.5),
    "Sichuan_China":    (102.0, 29.5, 105.5, 32.5),
    "W_Australia":      (117.0, -32.0, 120.5, -29.0),
    "S_Norway":         (5.0, 58.5, 9.0, 61.5),
    "Ordos_China":      (107.5, 37.0, 111.0, 40.0),
}


print(f"\n{'Patch':<25} {'N cells':>7} {'Min km':>8} {'Max km':>8} {'Mean km':>8}")
print("-" * 60)
for name, (minlon, minlat, maxlon, maxlat) in patches.items():
    patch = df[
        (df['lon'] >= minlon) & (df['lon'] <= maxlon) &
        (df['lat'] >= minlat) & (df['lat'] <= maxlat)
    ]
    if len(patch) > 0:
        t = patch['thickness_km']
        print(f"{name:<25} {len(patch):>7} {t.min():>8.2f} {t.max():>8.2f} {t.mean():>8.2f}")
    else:
        print(f"{name:<25}  NO DATA IN BOUNDS")

Zero/negative thickness locations: 4150
  Lat range of zeros: -86.5 – 82.5
  Lon range of zeros: -179.5 – 179.5

Patch                     N cells   Min km   Max km  Mean km
------------------------------------------------------------
Kanto_Japan                    12     0.10     2.50     0.88
Tohoku_Japan                   16     0.00     2.00     0.68
Central_Chile                  16     0.05     3.00     0.52
Central_Turkey                 12     0.00     3.50     0.98
Nepal                          12     0.00     5.00     0.67
North_Island_NZ                16     0.20     3.90     1.52
Sumatra                        20     0.20     3.70     1.58
Kutch_India                    16     0.01     4.00     1.08
Sichuan_China                  16     0.00     9.00     3.56
W_Australia                    12     0.00     0.00     0.00
S_Norway                       16     0.00     0.65     0.05
Ordos_China                    12     1.30     7.20     3.86


In [7]:
patch_means = {
    "Kanto_Japan":   (0.88, 532.5),   # (sediment_km, vs30)
    "W_Australia":   (0.00, 441.4),
    "Central_Chile": (0.52, 622.3),
    "Ordos_China":   (3.86, 354.4),
    "Sichuan_China": (3.56, None),    # vs30 not yet checked
    "S_Norway":      (0.05, None),
}

print("Patch-level sediment vs Vs30 (sanity check):")
print(f"{'Patch':<20} {'Sediment km':>12} {'Vs30 m/s':>10} {'Expected':>12}")
print("-" * 58)
for name, (sed, vs30) in patch_means.items():
    if vs30:
        expected = "anticorrelated" if (sed > 1 and vs30 < 500) or (sed < 1 and vs30 > 500) else "CHECK"
        print(f"{name:<20} {sed:>12.2f} {vs30:>10.1f} {expected:>12}")

Patch-level sediment vs Vs30 (sanity check):
Patch                 Sediment km   Vs30 m/s     Expected
----------------------------------------------------------
Kanto_Japan                  0.88      532.5 anticorrelated
W_Australia                  0.00      441.4        CHECK
Central_Chile                0.52      622.3 anticorrelated
Ordos_China                  3.86      354.4 anticorrelated


## Insights

<p>The sediment thickness layer is sourced from CRUST1.0, a global 1° resolution crustal model provided as an XYZ point file covering 64,800 grid cells (360×180) from 89.5°S to 89.5°N. Values are stored in metres and range from 0 to 21,000 m (0–21 km), consistent with known maximum sedimentary basin depths in regions like the Caspian Sea and Gulf of Mexico. All values were converted to kilometres for interpretability and consistency with the literature.</p>

<p>The 4,150 zero-value cells are distributed across all latitudes and longitudes, confirming they represent exposed basement outcrops globally rather than missing data. These are treated as genuine zero sediment thickness values throughout the analysis. One notable case is Western Australia, where all 12 CRUST1.0 cells within the patch return exactly 0.00 km. This is almost certainly a limitation of CRUST1.0's representation of thin surficial sediment on exposed Archean craton rather than a true absence of any sediment cover. This is documented as a known dataset limitation but does not affect the analysis, as zero sediment thickness combined with the patch's other static features (moderate Vs30, zero fault density) produces a coherent stable craton signature for the geological clustering step.</p>

<p>Patch-level mean sediment thicknesses are physically interpretable and ordered as expected. Ordos China (3.86 km) and Sichuan (3.56 km) record the highest values, reflecting the thick Loess Plateau cover and deep Sichuan Basin sedimentary sequence respectively. New Zealand North Island (1.52 km) and Sumatra (1.58 km) show moderate thickness consistent with active margin basin settings. Norway (0.05 km) and Chile (0.52 km) are lowest among the well-instrumented patches, reflecting exposed Baltic Shield and predominantly hard Andean rock respectively.</p>

<p>The fundamental limitation of this layer is its coarse 1° resolution. Each patch contains only 12–20 CRUST1.0 cells, meaning interpolation to the 0.1° processing grid will spread a small number of point values across approximately 900 cells per patch. This introduces significant smoothing and will suppress sub-crustal heterogeneity that finer-resolution datasets would capture. This limitation is explicitly acknowledged in the paper Methods section and motivates the inclusion of the higher-resolution Vs30 and DEM layers as complementary static features that capture surface and near-surface geological variability at finer spatial scales.</p>

<p>A patch-level anticorrelation check between sediment thickness and Vs30 confirms the expected physical relationship, thicker sediments correspond to lower shear wave velocities, for three of the four patches tested (Kanto, Chile, Ordos). The Western Australia exception is consistent with the known Vs30 underestimation on flat cratonic terrain identified during the Vs30 inspection, and the two anomalies are therefore coherent rather than independently concerning. This cross-layer consistency provides confidence that both datasets are capturing real geological signal and that their combination in the frozen prior will produce meaningful regime discrimination.</p>